# Algorithmic Lab: Paradigms, Dynamic Time Warping, and Unit Tests

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. Algorithmic Design Paradigms Comparison

Across algorithmic computer science and AI system engineering, five fundamental design paradigms recur:
1. **Brute Force / Exhaustive Search:** Evaluates all states in the search space (e.g., linear search, TSP permutation search). Guaranteed optimal but exponential.
2. **Recursion:** Directly expresses problem solutions in terms of self-similar smaller instances via the runtime call stack.
3. **Two Pointers:** Exploits monotonic structure, sorting, or directional symmetry to eliminate nested scans ($O(n^2) \to O(n)$).
4. **Divide-and-Conquer:** Partitions a problem into independent subproblems, solves them recursively, and combines results ($O(n \log n)$).
5. **Dynamic Programming:** Solves overlapping subproblems once, recording results in a memo or table ($O(2^n) \to O(n)$ or $O(n^2)$).


In [ ]:
from __future__ import annotations

import math
import unittest

## 2. Dynamic Time Warping (DTW) for Time-Series Alignment

In speech recognition, sensor telemetry, and motion analysis, two time series $X$ and $Y$ may have similar shape but varying temporal speeds. **Dynamic Time Warping (DTW)** uses 2D dynamic programming to find the optimal nonlinear alignment path that minimizes cumulative distance.


In [ ]:
def dynamic_time_warping(x: list[float], y: list[float]) -> tuple[float, list[tuple[int, int]]]:
    """Compute DTW alignment distance and optimal warping path."""
    n, m = len(x), len(y)
    cost = [[math.inf] * (m + 1) for _ in range(n + 1)]
    cost[0][0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d = abs(x[i - 1] - y[j - 1])
            cost[i][j] = d + min(
                cost[i - 1][j],  # Insertion
                cost[i][j - 1],  # Deletion
                cost[i - 1][j - 1],  # Match
            )

    # Reconstruct optimal warping path
    path = []
    i, j = n, m
    while i > 0 and j > 0:
        path.append((i - 1, j - 1))
        steps = [
            (cost[i - 1][j - 1], i - 1, j - 1),
            (cost[i - 1][j], i - 1, j),
            (cost[i][j - 1], i, j - 1),
        ]
        _, i, j = min(steps, key=lambda s: s[0])

    path.reverse()
    return cost[n][m], path


# Demonstrate on two temporal signals with phase shift
sig1 = [0.0, 1.0, 2.0, 1.0, 0.0]
sig2 = [0.0, 0.0, 1.0, 2.0, 1.0, 0.0]
dtw_dist, warp_path = dynamic_time_warping(sig1, sig2)
print(f"DTW Distance: {dtw_dist:.4f}")
print("Warping Path (sig1_idx, sig2_idx):", warp_path)

## 3. Comprehensive Session Test Suite


In [ ]:
class SessionAlgorithmTests(unittest.TestCase):
    def test_merge_sort(self):
        arr = [7, 2, 5, 30, 16, 8, 4, 17]

        def merge_sort_local(s):
            if len(s) < 2:
                return
            mid = len(s) // 2
            s1, s2 = s[:mid], s[mid:]
            merge_sort_local(s1)
            merge_sort_local(s2)
            i = j = 0
            while i + j < len(s):
                if j == len(s2) or (i < len(s1) and s1[i] <= s2[j]):
                    s[i + j] = s1[i]
                    i += 1
                else:
                    s[i + j] = s2[j]
                    j += 1

        merge_sort_local(arr)
        self.assertEqual(arr, [2, 4, 5, 7, 8, 16, 17, 30])

    def test_fibonacci_dp(self):
        dp = [0, 1]
        for i in range(2, 8):
            dp.append(dp[i - 1] + dp[i - 2])
        self.assertEqual(dp[7], 13)

    def test_lis_reconstruction(self):
        nums = [3, 10, 2, 1, 20]
        n = len(nums)
        dp = [1] * n
        trace = [-1] * n
        for i in range(n):
            for j in range(i):
                if nums[i] > nums[j] and dp[i] < dp[j] + 1:
                    dp[i] = dp[j] + 1
                    trace[i] = j
        self.assertEqual(max(dp), 3)
        idx = dp.index(3)
        path = []
        while idx != -1:
            path.append(nums[idx])
            idx = trace[idx]
        path.reverse()
        self.assertEqual(path, [3, 10, 20])


suite = unittest.defaultTestLoader.loadTestsFromTestCase(SessionAlgorithmTests)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful()